# Benchmark tune online cho phân cụm luồng (`conf_pipeline`)

Notebook này chạy trọn giao thức của `conf_pipeline` ở một chỗ. Mỗi bộ (thuật toán, luồng, seed) được chạy dưới bốn chế độ tune, gọi là *arm*, trên cùng một driver: `default` giữ một cấu hình default của River suốt luồng, `confstream` là ensemble confStream tự chọn lại cấu hình theo silhouette sau mỗi cửa sổ, `random_online` giống `confstream` nhưng rút cấu hình con đều trên không gian tìm kiếm, còn `frozen_prefix` tune như `confstream` rồi giữ nguyên cấu hình từ cửa sổ `FREEZE_AFTER_WINDOW`. Một lượt quét Sobol các cấu hình tĩnh chọn ra oracle tĩnh cho từng cặp (thuật toán, luồng), và oracle đó được chạy lại như arm thứ năm `oracle_static` để làm trần. Các cell cuối dựng báo cáo, phân tích thứ hạng giữa các arm và ước tính thời gian cho giao thức đầy đủ.

**Cách dùng.** Mở notebook từ gốc `river-customized-benchmark-framework/` rồi chạy lần lượt từ trên xuống. Preset được đọc từ biến môi trường `OTB_PRESET`:

| Preset | Dùng ở đâu | Quy mô |
|---|---|---|
| `tiny` | máy Mac, kiểm tra đường ống trong vài phút | DenStream × `static_easy` × seed 1 × 2 arm, 900 mẫu |
| `pilot` (mặc định) | máy Mac hoặc server | DenStream, CluStream, HPStream, RepStream × 4 luồng × seed 1 × 4 arm, 100 000 mẫu |
| `full` | server Linux | 8 thuật toán × 18 luồng × 5 seed × 4 arm, 100 000 mẫu |

Trên máy Mac có thể mở bằng `OTB_PRESET=tiny jupyter lab`. Trên server nên chạy trong `tmux`, ví dụ `OTB_PRESET=full jupyter nbconvert --to notebook --execute otb_benchmark.ipynb --output otb_full_run.ipynb --ExecutePreprocessor.timeout=-1`. Kết quả nằm ở `conf_pipeline/out/<preset>/` (đổi bằng `OTB_OUT_DIR`): `runs.jsonl`, `windows.jsonl`, `members/`, `sweep.jsonl`, `oracle_static.json` và thư mục `report/`.

**Mọi cell đều chạy lại được.** Job đã có dòng trong file kết quả được bỏ qua, nên khi dừng giữa chừng thì chỉ cần chạy lại cell để làm tiếp phần còn dở. Job lỗi không tự chạy lại; muốn thử lại chúng thì đặt `RETRY_FAILED = True` trong bảng điều khiển. Kết quả chạy trên server rồi `scp` về `conf_pipeline/out/<preset>/` được các cell báo cáo đọc ngay mà không phải chạy lại gì.

In [ ]:
from __future__ import annotations

import os
import signal
import subprocess
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

# The notebook lives at the framework root; conf_pipeline must be importable from here.
REPO = Path.cwd()
if not (REPO / "conf_pipeline").is_dir():
    raise SystemExit(f"Hãy mở notebook từ gốc river-customized-benchmark-framework (cwd={REPO})")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from conf_pipeline import _common as C, analyze, report  # noqa: E402

# Every algorithm run is a child process with the hash seed pinned (CluStream replays only then).
ENV = {**os.environ, "PYTHONHASHSEED": C.PYTHONHASHSEED}
# report/analyze switch pyplot to Agg to write their PNGs; their cells switch back to this.
BACKEND = matplotlib.get_backend()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def run_cli(module: str, out_dir: Path, *args) -> int:
    """``python -m <module> --out-dir <out_dir> <args>`` from the repo root, streaming its output.

    Same contract as ``subprocess.run(..., check=False)``: a non-zero exit is
    printed, not raised. Interrupting the cell stops the driver and its jobs
    (their own process group: SIGTERM, then SIGKILL if it is still running
    30 s later) and removes the lock files the driver held, so the cell can
    simply be run again: finished jobs are on file and skipped.
    """
    cmd = [sys.executable, "-m", module, "--out-dir", str(out_dir), *map(str, args)]
    print("$ PYTHONHASHSEED=0 python -m", *cmd[2:], flush=True)
    # Unbuffered, so lines the CLI prints without flushing (its job count) show up at once.
    proc = subprocess.Popen(cmd, cwd=REPO, env={**ENV, "PYTHONUNBUFFERED": "1"}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()
        for lock in Path(out_dir).glob("*.lock"):
            if lock.read_text(encoding="utf-8").startswith(f"pid={proc.pid} "):
                lock.unlink()
        print("Đã dừng; chạy lại cell để làm tiếp các job còn dở.")
        raise
    if rc != 0:
        print(f"!! {module} thoát với mã {rc}; các cell sau vẫn đọc được những gì đã ghi.")
    return rc


def show(title: str, df: pd.DataFrame, cols: list[str] | None = None) -> None:
    """A bold title, then ``df`` (only the ``cols`` it has), or a note when it is empty."""
    display(Markdown(f"**{title}**"))
    if df is None or df.empty:
        print("(trống)")
        return
    display(df[[c for c in cols if c in df.columns]] if cols else df)

## 1. Bảng điều khiển

Cell dưới chọn preset theo `OTB_PRESET` (mặc định `pilot`) và thư mục kết quả theo `OTB_OUT_DIR` (mặc định `conf_pipeline/out/<preset>`), dựng sẵn danh sách cờ cho ba lệnh CLI rồi in toàn bộ thiết lập đã chốt để đối chiếu trước khi chạy. Muốn đổi một giá trị cho riêng lần chạy này thì gán lại ở chỗ đánh dấu ngay dưới các cờ, ví dụ `WORKERS = 4`; khi đổi `N_SAMPLES`, `H`, `E_SIZE`, `E_TEST` hay `FREEZE_AFTER_WINDOW` thì nên đổi cả `OTB_OUT_DIR` để hai lần chạy không trộn vào cùng một file. `FREEZE_AFTER_WINDOW` là cửa sổ mà `frozen_prefix` thôi tune và cũng là nơi phần đuôi của mọi arm bắt đầu; `tiny` đặt nó bằng 1 vì luồng chỉ có 3 cửa sổ. `RUN_SWEEP` và `RUN_ORACLE` bật hoặc tắt hai bước quét và chạy lại oracle, `LOG_MEMBERS` ghi kết quả của từng thành viên ensemble (cần cho `R_proxy`, `R_search` và mốc hồi phục sau drift), còn `RETRY_FAILED = True` chạy lại các job đã lỗi.

In [ ]:
PRESET = os.environ.get("OTB_PRESET", "pilot")
OUT_DIR = Path(os.environ.get("OTB_OUT_DIR", REPO / "conf_pipeline" / "out" / PRESET))

PRESETS = {
    "tiny": dict(ALGOS=["DenStream"], FOLDS=["static_easy"], SEEDS=[1], ARMS=["default", "confstream"],
                 N_SAMPLES=900, H=300, E_SIZE=2, E_TEST=1, FREEZE_AFTER_WINDOW=1, SWEEP_N_POINTS=2,
                 WORKERS=1, TIMEOUT_S=600, ORACLE_SEEDS=[1]),
    "pilot": dict(ALGOS=["DenStream", "CluStream", "HPStream", "RepStream"],
                  FOLDS=["covtype", "insects_abrupt_balanced", "scale_drift", "k_drift"],
                  SEEDS=[1], ARMS=list(C.ARMS), N_SAMPLES=C.N_SAMPLES_CAP, H=C.H, E_SIZE=C.E_SIZE,
                  E_TEST=C.E_TEST, FREEZE_AFTER_WINDOW=C.FREEZE_AFTER_WINDOW, SWEEP_N_POINTS=8,
                  WORKERS=8, TIMEOUT_S=6 * 3600, ORACLE_SEEDS=[1]),
    "full": dict(ALGOS=list(C.ALGOS), FOLDS=list(C.FOLDS), SEEDS=list(C.SEEDS), ARMS=list(C.ARMS),
                 N_SAMPLES=C.N_SAMPLES_CAP, H=C.H, E_SIZE=C.E_SIZE, E_TEST=C.E_TEST,
                 FREEZE_AFTER_WINDOW=C.FREEZE_AFTER_WINDOW, SWEEP_N_POINTS=C.SWEEP_N_POINTS,
                 WORKERS=8, TIMEOUT_S=12 * 3600,
                 ORACLE_SEEDS=[2, 3, 4, 5, 6]),
}
if PRESET not in PRESETS:
    raise SystemExit(f"OTB_PRESET={PRESET!r}; chọn một trong {sorted(PRESETS)}")
globals().update(PRESETS[PRESET])
RUN_SWEEP = True
RUN_ORACLE = True
LOG_MEMBERS = True
RETRY_FAILED = False

# --- overrides for this run only go here, e.g. WORKERS = 4 ---

# --- derived: the flags of the three CLIs (nothing to edit below) ---
OUT_DIR.mkdir(parents=True, exist_ok=True)
COMMON_ARGS = ["--algos", ",".join(ALGOS), "--folds", ",".join(FOLDS), "--n-samples", N_SAMPLES,
               "--h", H, "--workers", WORKERS, "--timeout", TIMEOUT_S]
COMMON_ARGS += ["--retry-failed"] if RETRY_FAILED else []
DRIVER_ARGS = ["--e-size", E_SIZE, "--e-test", E_TEST, "--freeze-after-window", FREEZE_AFTER_WINDOW]
RUN_ARGS = [*COMMON_ARGS, *DRIVER_ARGS, "--seeds", ",".join(map(str, SEEDS)), "--arms", ",".join(ARMS)]
RUN_ARGS += ["--log-members"] if LOG_MEMBERS else []
SWEEP_ARGS = [*COMMON_ARGS, "--n-points", SWEEP_N_POINTS, "--seed", 1]
ORACLE_ARGS = [*COMMON_ARGS, *DRIVER_ARGS, "--seeds", ",".join(map(str, ORACLE_SEEDS)),
               "--arms", "oracle_static"]

print(f"{'PRESET':19s} = {PRESET}")
print(f"{'OUT_DIR':19s} = {OUT_DIR}")
for name in ("ALGOS", "FOLDS", "SEEDS", "ARMS", "N_SAMPLES", "H", "E_SIZE", "E_TEST",
             "FREEZE_AFTER_WINDOW", "SWEEP_N_POINTS", "ORACLE_SEEDS", "WORKERS", "TIMEOUT_S", "RUN_SWEEP",
             "RUN_ORACLE", "LOG_MEMBERS", "RETRY_FAILED"):
    print(f"{name:19s} = {globals()[name]}")

## 2. Chạy thử (dry run)

Cell dưới gọi `conf_pipeline.run` và `conf_pipeline.sweep` với `--dry-run`, nên không job nào thật sự chạy. Dòng đầu của `run` cho tổng số job, số job đã có trong `runs.jsonl` và số job còn phải chạy; dòng đầu của `sweep` là số điểm quét, tính cả điểm default của River và chưa trừ những điểm đã có trong `sweep.jsonl`. Vài job đầu được liệt kê bên dưới để đối chiếu với bảng điều khiển.

In [ ]:
SHOW = 5
for module, args in (("conf_pipeline.run", RUN_ARGS), ("conf_pipeline.sweep", SWEEP_ARGS)):
    cmd = [sys.executable, "-m", module, "--out-dir", str(OUT_DIR), *map(str, args), "--dry-run"]
    res = subprocess.run(cmd, cwd=REPO, env=ENV, capture_output=True, text=True, check=False)
    lines = res.stdout.splitlines()
    print(f"[{module}] " + (lines[0] if lines else "(không có output)"))
    for line in lines[1:1 + SHOW]:
        print(line)
    if len(lines) > 1 + SHOW:
        print(f"  ... và {len(lines) - 1 - SHOW} dòng nữa")
    if res.returncode != 0:
        print(res.stderr[-2000:])

## 3. Chạy các arm

Cell dưới chạy mọi job (thuật toán, luồng, seed, arm) của preset bằng `conf_pipeline.run`; mỗi job là một tiến trình con có hash seed cố định, giới hạn bộ nhớ và giới hạn thời gian `TIMEOUT_S`, và có `WORKERS` job chạy song song. Mỗi dòng in ra là một job vừa xong, với trạng thái (`ok`, `error`, `timeout`, `oom`), số cửa sổ, ARI lagged trung bình trên toàn luồng và thời gian chạy. Kết quả được ghi dần vào `runs.jsonl` và `windows.jsonl` (cùng `members/` khi `LOG_MEMBERS` bật), nên dừng cell rồi chạy lại thì các job đã xong được bỏ qua.

In [ ]:
rc = run_cli("conf_pipeline.run", OUT_DIR, *RUN_ARGS)

## 4. Quét cấu hình tĩnh và chọn oracle

Khi `RUN_SWEEP` bật, cell dưới chạy `conf_pipeline.sweep`: với mỗi cặp (thuật toán, luồng), từng điểm Sobol của không gian tìm kiếm và điểm default của River được chạy một lần với seed 1 như một cấu hình tĩnh không bao giờ đổi. Cuối lượt quét, điểm có ARI lagged trung bình cao nhất trên toàn luồng được ghi vào `oracle_static.json`; điểm lỗi vẫn nằm trong `sweep.jsonl` nhưng không bao giờ được chọn. Khi `RUN_SWEEP` tắt, cell không làm gì và `oracle_static.json` đang có được giữ nguyên.

In [ ]:
if RUN_SWEEP:
    rc = run_cli("conf_pipeline.sweep", OUT_DIR, *SWEEP_ARGS)
else:
    print("RUN_SWEEP = False: bỏ qua lượt quét; oracle_static.json giữ nguyên.")

## 5. Chạy lại oracle tĩnh

Khi `RUN_ORACLE` bật, cell dưới chạy `conf_pipeline.run --arms oracle_static` với các seed trong `ORACLE_SEEDS`: mỗi cặp (thuật toán, luồng) chạy lại đúng cấu hình mà `oracle_static.json` đã chọn, như một arm một thành viên không đổi. Ở preset `full` các seed này (2 đến 6) tách khỏi seed 1 của lượt quét, nên trần không được đo trên chính lần chạy đã chọn ra nó; ở `tiny` và `pilot` trần dùng lại seed 1. Cặp nào chưa có oracle được ghi là `skipped` kèm lý do, vì vậy cell này phải chạy sau bước quét.

In [ ]:
if RUN_ORACLE:
    rc = run_cli("conf_pipeline.run", OUT_DIR, *ORACLE_ARGS)
else:
    print("RUN_ORACLE = False: bỏ qua; headroom lấy điểm tốt nhất của lượt quét làm trần.")

## 6. Báo cáo

Cell dưới gọi `report.write_report`, ghi `report.md` cùng các CSV và hình đường cong theo cửa sổ vào `OUT_DIR/report/`, rồi hiện các bảng chính ngay trong notebook. Mọi con số chất lượng là ARI *lagged*: cấu hình đương nhiệm sau cửa sổ t−1 được chấm trên cửa sổ t. Các bảng lần lượt là tóm tắt theo (thuật toán, luồng, arm), chênh lệch so với `default`, trần tĩnh và headroom, phân rã regret, thích nghi với drift và Table A; cách đọc từng bảng nằm ở phần ghi chú cuối notebook. Preset `tiny` chỉ có 3 cửa sổ nên đặt `FREEZE_AFTER_WINDOW = 1`, để phần đuôi gồm các cửa sổ từ 1 trở đi; con số của nó chỉ dùng để kiểm tra đường ống, không để đọc kết quả.

In [ ]:
written = report.write_report(OUT_DIR)
plt.switch_backend(BACKEND)
print(f"Đã ghi {len(written)} file vào {OUT_DIR / 'report'}")

summary = report.summary_table(OUT_DIR)
show("Tóm tắt: trung bình qua seed", summary,
     ["algo", "fold", "arm", "n_seeds", "lag_ari_win_tail_mean", "lag_ari_win_tail_sd",
      "incumbent_throughput_sps"])
show("Arm so với default", report.gain_table(OUT_DIR))
show("Trần tĩnh và headroom", report.headroom_table(OUT_DIR))
regret = report.regret_table(OUT_DIR)
show("Phân rã regret trên phần đuôi (region = tail)", regret[regret["region"] == "tail"])
show("Thích nghi với drift", report.drift_table(OUT_DIR))
show("Table A: phân cụm lại chung trên micro-cluster", summary,
     ["algo", "fold", "arm", "n_seeds", *report.TABLE_A_FIELDS])

## 7. Phân tích giữa các arm

Cell dưới gọi `analyze.write_analysis`, ghi `analysis.md`, các CSV và hình F1–F4 vào `OUT_DIR/report/`, rồi hiện các bảng tính trên cùng độ đo `lag_ari_win_tail_mean`. Bảng đầu cho hạng trung bình của từng thuật toán dưới mỗi arm (hạng 1 là tốt nhất) kèm p của kiểm định Friedman và khoảng cách tới hạn `cd` của Nemenyi, còn bảng thứ hai là ma trận Kendall τ giữa thứ hạng thuật toán của các arm. Sau đó là số luồng mà `confstream` thắng, hoà hoặc thua `default` với mỗi thuật toán, bảng Pareto giữa chất lượng và thông lượng, và cuối cùng là các hình. Preset `tiny` chỉ có một thuật toán nên không có thứ hạng nào để so, vì vậy Kendall τ ngoài đường chéo, p của Friedman và `cd` để trống.

In [ ]:
written = analyze.write_analysis(OUT_DIR)
plt.switch_backend(BACKEND)
print(f"Đã ghi {len(written)} file; tóm tắt ở {OUT_DIR / 'report' / 'analysis.md'}")

summary = report.summary_table(OUT_DIR)
present = set(summary["arm"]) if "arm" in summary else set()
arms = [a for a in analyze.ARM_ORDER if a in present] + sorted(present - set(analyze.ARM_ORDER))
if not arms:
    print("Chưa có run nào để phân tích.")
else:
    rows = []
    for arm in arms:
        fn = analyze.friedman_nemenyi(analyze.rank_matrix(summary, arm))
        rows.append({"arm": arm, **fn["avg_rank"].to_dict(), "n_folds": fn["n_folds"],
                     "friedman_p": fn["pvalue"], "cd": fn["cd"]})
    show("Hạng trung bình theo arm (1 = tốt nhất) và CD Nemenyi", pd.DataFrame(rows).set_index("arm"))
    show("Kendall τ giữa thứ hạng thuật toán của các arm", analyze.kendall_matrix(summary, arms))
    if {"default", "confstream"} <= present:
        show("confstream so với default: thắng / hoà / thua theo luồng",
             analyze.win_tie_loss(summary, "confstream"))
    show("Pareto: ARI tail và thông lượng", analyze.pareto_table(summary))
for path in written:
    if path.suffix == ".png":
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))

## 8. Ngân sách thời gian

Cell dưới đọc `runs.jsonl`, đếm job theo arm và trạng thái, rồi tính `wall_s` trung bình của các job `ok` theo (thuật toán, arm). Từ `wall_s` trung bình của mỗi arm, bảng cuối ngoại suy ra giao thức đầy đủ (8 thuật toán × 18 luồng × 5 seed × 4 arm), cộng thêm lượt quét và lần chạy lại oracle khi đã có số đo của chúng, với giả định luôn có `WORKERS` job chạy song song. Con số này là ước lượng thô, vì trung bình lấy trên các thuật toán và luồng của preset hiện tại và phép chia cho `WORKERS` bỏ qua thời gian chờ các job chậm nhất. Nó chỉ có nghĩa khi `N_SAMPLES` bằng mức cắt 100 000 của giao thức, tức là với `pilot` hoặc `full`.

In [ ]:
runs = pd.DataFrame([r for r in C.read_jsonl(OUT_DIR / "runs.jsonl") if r.get("kind") == "run"])
if runs.empty:
    print(f"Chưa có dòng nào trong {OUT_DIR / 'runs.jsonl'}.")
else:
    runs["wall_s"] = pd.to_numeric(runs.get("wall_s"), errors="coerce")
    ok = runs[runs["status"] == "ok"]
    show("Số job theo arm và trạng thái", runs.groupby(["arm", "status"]).size().unstack(fill_value=0))
    if not ok.empty:
        show("wall_s trung bình (giây) của job ok, theo (algo, arm)",
             ok.pivot_table(index="algo", columns="arm", values="wall_s", aggfunc="mean").round(1))

    full = PRESETS["full"]
    n_cells = len(full["ALGOS"]) * len(full["FOLDS"])
    mean_wall = ok.groupby("arm")["wall_s"].mean()
    sweep_wall = [r.get("wall_s") for r in C.read_jsonl(OUT_DIR / "sweep.jsonl") if r.get("status") == "ok"]
    plan = [(arm, mean_wall.get(arm), n_cells * len(full["SEEDS"])) for arm in full["ARMS"]]
    plan.append(("oracle_static", mean_wall.get("oracle_static"), n_cells * len(full["ORACLE_SEEDS"])))
    plan.append(("sweep", pd.Series(sweep_wall, dtype=float).mean(), n_cells * (full["SWEEP_N_POINTS"] + 1)))
    budget = pd.DataFrame(plan, columns=["step", "mean_wall_s", "n_jobs_full"]).set_index("step")
    budget["mean_wall_s"] = budget["mean_wall_s"].astype(float)
    budget["cpu_h"] = budget["mean_wall_s"] * budget["n_jobs_full"] / 3600
    budget["wall_h"] = budget["cpu_h"] / WORKERS
    show(f"Ngoại suy ra giao thức đầy đủ (wall_h với {WORKERS} worker)", budget.round(2))

    arms4 = budget.loc[list(full["ARMS"])]
    print(f"4 arm, {arms4['n_jobs_full'].sum()} job: ~{arms4['cpu_h'].sum():.1f} giờ CPU, "
          f"~{arms4['wall_h'].sum():.1f} giờ với {WORKERS} worker")
    print(f"Tính cả sweep và oracle: ~{budget['cpu_h'].sum():.1f} giờ CPU, "
          f"~{budget['wall_h'].sum():.1f} giờ với {WORKERS} worker")
    missing = budget.index[budget["mean_wall_s"].isna()].tolist()
    if missing:
        print(f"Chưa có số đo cho {missing}: các dòng này không được cộng vào tổng.")
    if N_SAMPLES != C.N_SAMPLES_CAP:
        print(f"Preset này chạy N_SAMPLES = {N_SAMPLES}, khác mức cắt {C.N_SAMPLES_CAP} của giao thức "
              "đầy đủ, nên phép ngoại suy ở đây chỉ để kiểm tra cách tính.")

## 9. Ghi chú: cách đọc từng bảng

**Tóm tắt.** Mỗi dòng là một bộ (thuật toán, luồng, arm), lấy trung bình qua các seed đã chạy xong (`n_seeds`), và cột `_sd` là độ lệch chuẩn giữa các seed. Mọi cột chất lượng là ARI lagged, tức cấu hình đương nhiệm sau cửa sổ t−1 được chấm trên cửa sổ t, nên không con số nào là điểm của một cấu hình trên chính cửa sổ đã chọn ra nó. Cột có `tail` chỉ lấy các cửa sổ từ `FREEZE_AFTER_WINDOW` trở đi (20 ở `pilot` và `full`, 1 ở `tiny`), vùng mà mọi arm cùng có và `frozen_prefix` đã thôi tune, nên đây là con số chính để so các arm; cột không có `tail` lấy toàn luồng, gồm cả giai đoạn đầu khi ensemble còn đang dò. `incumbent_throughput_sps` là thông lượng (mẫu mỗi giây) của riêng cấu hình đương nhiệm, khác với chi phí của cả ensemble ở `ensemble_runtime_s`.

**Arm so với default.** `gain_tail` và `gain_all` là ARI lagged của arm trừ đi của `default`, trên phần đuôi và trên toàn luồng. Gain dương trên đuôi nhưng âm trên toàn luồng nghĩa là arm trả giá ở giai đoạn dò rồi mới vượt default. `throughput_ratio` là thông lượng của cấu hình đương nhiệm của arm chia cho của `default`, nên nhỏ hơn 1 nghĩa là cấu hình được tune chạy chậm hơn default.

**Trần tĩnh và headroom.** Trần (`ceiling`) là ARI lagged của oracle tĩnh: trung bình các lần chạy lại `oracle_static` khi có, nếu không thì điểm tốt nhất của lượt quét (`ceiling_source` cho biết nguồn, `metric` cho biết đo trên đuôi hay toàn luồng). Một cặp chỉ được coi là tune được (`tunable`) khi `gain = ceiling − default` đạt ít nhất `MIN_CEILING_GAIN` = 0,05 và lớn hơn 1,645 lần sai số chuẩn của default qua seed; với một seed thì sai số chuẩn bằng 0 và cổng chỉ còn điều kiện 0,05. Cặp không qua cổng để trống mọi cột headroom và ghi lý do ở `note`, thay vì đưa ra một tỉ số của nhiễu. Với cặp tune được, `headroom_<arm> = (arm − default) / gain`, trong đó 0 là ngang default, 1 là đạt trần tĩnh, số âm là tệ hơn default và số lớn hơn 1 là tune online vượt cả cấu hình tĩnh tốt nhất; `evolution_value = headroom_confstream − headroom_random_online` là phần đóng góp riêng của cơ chế tiến hoá so với rút cấu hình ngẫu nhiên.

**Phân rã regret.** Khoảng cách tới trần được tách thành `A_static − a_lag = R_lag + R_proxy + R_search`, lấy trung bình theo cửa sổ rồi theo seed. Bảng trong notebook chỉ hiện vùng đuôi (`region = tail`), tức các cửa sổ từ `tail_from` trở đi, và ở vùng này `A_static` là ARI trên đuôi của oracle tĩnh; `regret.csv` và `report.md` có thêm dòng `region = all` cho toàn luồng, khi đó `A_static` là ARI toàn luồng của oracle. Cột `ceiling_source` cho biết trần lấy từ các lần chạy lại oracle hay từ lượt quét. `R_lag = a_inc − a_lag` là cái giá của việc chậm một cửa sổ, tức cấu hình chọn sau cửa sổ t−1 chấm trên cửa sổ t so với cấu hình chọn ngay trên cửa sổ t. `R_proxy = a_star − a_inc` là cái giá của việc chọn bằng silhouette thay vì ARI, với `a_star` là ARI của thành viên tốt nhất đang hoạt động trên cửa sổ; `R_search = A_static − a_star` là phần mà ensemble chưa bao giờ giữ được cấu hình tĩnh tối ưu, và giá trị âm nghĩa là thích nghi đã vượt cấu hình tĩnh tốt nhất. `R_proxy` và `R_search` cần log thành viên (`LOG_MEMBERS = True`), còn `R_search` cần thêm oracle tĩnh; thiếu thì cột để trống và `note` nói lý do.

**Thích nghi với drift.** Bảng chỉ có dòng cho các luồng tổng hợp có điểm drift (`scale_drift`, `density_drift`, `k_drift`, `mixed_drift`). `pre` là ARI lagged ở cửa sổ ngay trước drift, và `drop_depth` là `pre` trừ ARI lagged thấp nhất trong 10 cửa sổ kể từ drift, tức độ sâu của cú rơi. `recovery_windows` là số cửa sổ cho tới khi ARI lagged nằm trong khoảng 0,05 dưới mốc `ref` suốt 3 cửa sổ liên tiếp, và để trống nếu không hồi phục trước drift kế tiếp; `ref` là thành viên tốt nhất đang hoạt động khi có log thành viên và hơn một thành viên, nếu không thì là `pre`. `param_ratio` chỉ có ở `scale_drift` và `mixed_drift`: đó là tham số khoảng cách của cấu hình đương nhiệm sau khi hồi phục chia cho giá trị trước drift, nên một bộ tune bám được thang dữ liệu sẽ cho tỉ số gần hệ số nhân của thang.

**Table A.** Mỗi thuật toán luồng gộp micro-cluster thành cụm bằng pha offline riêng, nên chênh lệch ARI giữa hai thuật toán trộn lẫn pha online với pha offline. Table A đưa micro-cluster của cấu hình lagged của mọi thuật toán qua cùng một bộ phân cụm lại, gồm k-means với k bằng số lớp thật (`lag_ari_A_kmeans_*`) và DBSCAN với `eps = 0,5·s` (`lag_ari_A_dbscan_*`), nên phần chênh lệch còn lại giữa các thuật toán là của pha online. `lag_n_micro_centers_mean` là số micro-cluster, còn `lag_ssq_micro_mean` là tổng bình phương khoảng cách từ các điểm của cửa sổ tới tâm micro-cluster gần nhất.

**Thứ hạng và Kendall τ.** Trên mỗi luồng, các thuật toán được xếp hạng theo `lag_ari_win_tail_mean` dưới từng arm, và thuật toán không có số đo trên một luồng nhận hạng kém nhất. Một ô chỉ được tính khi mọi seed mà arm đó đã chạy đều xong (`all_seeds_ok`); ô thiếu seed bị coi như không có số đo, nên nhận hạng kém nhất và không vào Kendall τ, bảng thắng/hoà/thua hay bảng Pareto. Bảng hạng trung bình đi kèm p của kiểm định Friedman và CD Nemenyi: hai thuật toán có hạng trung bình lệch nhau hơn CD thì khác nhau ở mức α = 0,05. Kendall τ giữa hai arm là trung bình qua các luồng của τ-b giữa hai vector hạng, với τ = 1 là cùng thứ tự, 0 là không liên quan và −1 là đảo ngược. Đây là con số chính của bài báo: τ giữa `default` và các arm tune online thấp hơn hẳn 1 nghĩa là thứ hạng thuật toán mà một benchmark công bố phụ thuộc vào chế độ tune, còn τ gần 1 nghĩa là so sánh ở default là đủ.

**Thắng, hoà, thua.** Với mỗi thuật toán, trên các luồng mà cả `confstream` và `default` đều có số đo, `confstream` thắng khi chênh lệch lớn hơn dải nhiễu `max(sd_arm, sd_default)` giữa các seed, thua khi nhỏ hơn âm của dải đó, và hoà trong các trường hợp còn lại. Với một seed thì dải nhiễu bằng 0, nên mọi chênh lệch khác 0 đều thành thắng hoặc thua, và bảng này chỉ đáng đọc khi có từ hai seed trở lên. `worst_gap` là chênh lệch âm nhất.

**Pareto.** `ari` là trung bình qua các luồng của ARI lagged trên đuôi, còn `sps` là thông lượng trung bình của cấu hình đương nhiệm. `hv` được tính trên từng luồng: ARI và log10 thông lượng được chuẩn hoá min–max trên mọi cặp (thuật toán, arm) của luồng đó, điểm của cặp chiếm một diện tích trong hình vuông đơn vị, và `hv` là trung bình diện tích ấy qua các luồng. `hv` càng gần 1 thì cặp đó càng vừa tốt vừa nhanh so với các cặp khác trên cùng luồng.